In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
DATA_PATH = Path("../data/interim/weekly_demand_500_articles.csv")

weekly_dataset = pd.read_csv(
    DATA_PATH,
    dtype={"article_id": "string"}
)

weekly_dataset["week_start"] = pd.to_datetime(
    weekly_dataset["week_start"]
)

weekly_dataset = weekly_dataset.sort_values(
    ["article_id", "week_start"]
).reset_index(drop=True)

print("Dataset loaded successfully!")

print("Total records:", len(weekly_dataset))
print("Unique articles:", weekly_dataset["article_id"].nunique())

display(weekly_dataset.head())

Dataset loaded successfully!
Total records: 20308
Unique articles: 500


,article_id,week_start,weekly_demand,product_type_name,product_group_name,colour_group_name,department_name,index_name,section_name,garment_group_name
0,0189616032,2019-12-09,20,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
1,0189616032,2019-12-16,45,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
2,0189616032,2019-12-23,38,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
3,0189616032,2019-12-30,29,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
4,0189616032,2020-01-06,32,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic


In [3]:
FORECAST_HORIZONS = [1, 4, 8]

TRAIN_END = pd.Timestamp("2020-02-17")

VALIDATION_START = pd.Timestamp("2020-02-24")
VALIDATION_END = pd.Timestamp("2020-06-08")

TEST_START = pd.Timestamp("2020-06-15")
TEST_END = pd.Timestamp("2020-09-21")

print("Forecast horizons:", FORECAST_HORIZONS)

print("Training ends:", TRAIN_END)
print("Validation:", VALIDATION_START, "to", VALIDATION_END)
print("Testing:", TEST_START, "to", TEST_END)

Forecast horizons: [1, 4, 8]
Training ends: 2020-02-17 00:00:00
Validation: 2020-02-24 00:00:00 to 2020-06-08 00:00:00
Testing: 2020-06-15 00:00:00 to 2020-09-21 00:00:00


In [4]:
def create_historical_features(df):

    df = df.copy()

    df = df.sort_values(
        ["article_id", "week_start"]
    )

    grouped_demand = df.groupby("article_id")["weekly_demand"]

    # Latest available demand at the forecast origin
    df["origin_demand"] = df["weekly_demand"]

    # Historical lag features
    df["demand_1w_before_origin"] = grouped_demand.shift(1)

    df["demand_2w_before_origin"] = grouped_demand.shift(2)

    df["demand_4w_before_origin"] = grouped_demand.shift(4)

    df["demand_8w_before_origin"] = grouped_demand.shift(8)

    # Rolling statistics using past and current observed demand
    df["rolling_mean_4"] = (
        grouped_demand
        .transform(
            lambda x: x.rolling(window=4).mean()
        )
    )

    df["rolling_mean_8"] = (
        grouped_demand
        .transform(
            lambda x: x.rolling(window=8).mean()
        )
    )

    df["rolling_std_4"] = (
        grouped_demand
        .transform(
            lambda x: x.rolling(window=4).std()
        )
    )

    return df


feature_dataset = create_historical_features(
    weekly_dataset
)

print("Historical demand features created!")

display(feature_dataset.head(15))

Historical demand features created!


,article_id,week_start,weekly_demand,product_type_name,product_group_name,colour_group_name,department_name,index_name,section_name,garment_group_name,origin_demand,demand_1w_before_origin,demand_2w_before_origin,demand_4w_before_origin,demand_8w_before_origin,rolling_mean_4,rolling_mean_8,rolling_std_4
0,0189616032,2019-12-09,20,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0189616032,2019-12-16,45,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic,45,20.0,NaN,NaN,NaN,NaN,NaN,NaN
2,0189616032,2019-12-23,38,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic,38,45.0,20.0,NaN,NaN,NaN,NaN,NaN
3,0189616032,2019-12-30,29,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic,29,38.0,45.0,NaN,NaN,33.00,NaN,10.862780
4,0189616032,2020-01-06,32,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic,32,29.0,38.0,20.0,NaN,36.00,NaN,7.071068
5,0189616032,2020-01-13,44,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic,44,32.0,29.0,45.0,NaN,35.75,NaN,6.652067
6,0189616032,2020-01-20,29,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic,29,44.0,32.0,38.0,NaN,33.50,NaN,7.141428
7,0189616032,2020-01-27,20,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic,20,29.0,44.0,29.0,NaN,31.25,32.125,9.912114
8,0189616032,2020-02-03,18,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic,18,20.0,29.0,32.0,20.0,27.75,31.875,11.842719
9,0189616032,2020-02-10,12,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic,12,18.0,20.0,44.0,45.0,19.75,27.750,7.041543


In [5]:
def create_forecast_target(df, horizon):

    df = df.copy()

    df = df.sort_values(
        ["article_id", "week_start"]
    )

    grouped = df.groupby("article_id")

    df["target_demand"] = (
        grouped["weekly_demand"].shift(-horizon)
    )

    df["target_week"] = (
        grouped["week_start"].shift(-horizon)
    )

    df["forecast_horizon"] = horizon

    return df


print("Forecast target function created!")

Forecast target function created!


In [8]:
def create_calendar_features(df):

    df = df.copy()

    # Calendar information based on forecast target date
    df["target_year"] = df["target_week"].dt.year

    df["target_month"] = df["target_week"].dt.month

    # Use nullable integer type to preserve missing dates
    df["target_week_of_year"] = (
        df["target_week"]
        .dt.isocalendar()
        .week
        .astype("Int64")
    )

    df["target_quarter"] = df["target_week"].dt.quarter

    return df


print("Calendar feature function created successfully!")

Calendar feature function created successfully!


In [9]:
def build_forecasting_dataset(base_dataset, horizon):

    # Create future target
    df = create_forecast_target(
        base_dataset,
        horizon
    )

    # Add calendar features
    df = create_calendar_features(df)

    # Select model input features
    feature_columns = [
        "origin_demand",
        "demand_1w_before_origin",
        "demand_2w_before_origin",
        "demand_4w_before_origin",
        "demand_8w_before_origin",
        "rolling_mean_4",
        "rolling_mean_8",
        "rolling_std_4",
        "target_month",
        "target_week_of_year",
        "target_quarter",
        "product_type_name",
        "product_group_name",
        "colour_group_name",
        "department_name",
        "index_name",
        "section_name",
        "garment_group_name"
    ]

    # Keep the necessary columns
    selected_columns = [
        "article_id",
        "week_start",
        "target_week",
        "target_demand",
        "forecast_horizon"
    ] + feature_columns

    df = df[selected_columns].copy()

    # Remove rows without sufficient historical features
    # or an observed future target
    df = df.dropna(
        subset=feature_columns + [
            "target_demand",
            "target_week"
        ]
    ).copy()

    df["target_demand"] = df["target_demand"].astype(int)

    return df, feature_columns


forecast_datasets = {}

for horizon in FORECAST_HORIZONS:

    horizon_dataset, feature_columns = build_forecasting_dataset(
        feature_dataset,
        horizon
    )

    forecast_datasets[horizon] = horizon_dataset

    print(f"\n{horizon}-Week Forecast Dataset")
    print("Records:", len(horizon_dataset))
    print("Unique articles:", horizon_dataset["article_id"].nunique())

    display(horizon_dataset.head(3))


1-Week Forecast Dataset
Records: 16276
Unique articles: 420


,article_id,week_start,target_week,target_demand,forecast_horizon,origin_demand,demand_1w_before_origin,demand_2w_before_origin,demand_4w_before_origin,demand_8w_before_origin,rolling_mean_4,rolling_mean_8,rolling_std_4,target_month,target_week_of_year,target_quarter,product_type_name,product_group_name,colour_group_name,department_name,index_name,section_name,garment_group_name
8,0189616032,2020-02-03,2020-02-10,12,1,18,20.0,29.0,32.0,20.0,27.75,31.875,11.842719,2.0,7,1.0,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
9,0189616032,2020-02-10,2020-02-17,12,1,12,18.0,20.0,44.0,45.0,19.75,27.750,7.041543,2.0,8,1.0,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
10,0189616032,2020-02-17,2020-02-24,23,1,12,12.0,18.0,29.0,38.0,15.50,24.500,4.123106,2.0,9,1.0,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic



4-Week Forecast Dataset
Records: 15032
Unique articles: 407


,article_id,week_start,target_week,target_demand,forecast_horizon,origin_demand,demand_1w_before_origin,demand_2w_before_origin,demand_4w_before_origin,demand_8w_before_origin,rolling_mean_4,rolling_mean_8,rolling_std_4,target_month,target_week_of_year,target_quarter,product_type_name,product_group_name,colour_group_name,department_name,index_name,section_name,garment_group_name
8,0189616032,2020-02-03,2020-03-02,18,4,18,20.0,29.0,32.0,20.0,27.75,31.875,11.842719,3.0,10,1.0,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
9,0189616032,2020-02-10,2020-03-09,8,4,12,18.0,20.0,44.0,45.0,19.75,27.750,7.041543,3.0,11,1.0,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
10,0189616032,2020-02-17,2020-03-16,3,4,12,12.0,18.0,29.0,38.0,15.50,24.500,4.123106,3.0,12,1.0,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic



8-Week Forecast Dataset
Records: 13437
Unique articles: 384


,article_id,week_start,target_week,target_demand,forecast_horizon,origin_demand,demand_1w_before_origin,demand_2w_before_origin,demand_4w_before_origin,demand_8w_before_origin,rolling_mean_4,rolling_mean_8,rolling_std_4,target_month,target_week_of_year,target_quarter,product_type_name,product_group_name,colour_group_name,department_name,index_name,section_name,garment_group_name
8,0189616032,2020-02-03,2020-03-30,1,8,18,20.0,29.0,32.0,20.0,27.75,31.875,11.842719,3.0,14,1.0,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
9,0189616032,2020-02-10,2020-04-06,0,8,12,18.0,20.0,44.0,45.0,19.75,27.750,7.041543,4.0,15,2.0,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
10,0189616032,2020-02-17,2020-04-13,0,8,12,12.0,18.0,29.0,38.0,15.50,24.500,4.123106,4.0,16,2.0,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic


In [10]:
def split_forecasting_dataset(df):

    train = df[
        df["target_week"] <= TRAIN_END
    ].copy()

    validation = df[
        (df["target_week"] >= VALIDATION_START)
        & (df["target_week"] <= VALIDATION_END)
    ].copy()

    test = df[
        (df["target_week"] >= TEST_START)
        & (df["target_week"] <= TEST_END)
    ].copy()

    return train, validation, test


forecast_splits = {}

for horizon, df in forecast_datasets.items():

    train, validation, test = split_forecasting_dataset(df)

    forecast_splits[horizon] = {
        "train": train,
        "validation": validation,
        "test": test
    }

    print(f"\n{'='*45}")
    print(f"{horizon}-WEEK FORECASTING")
    print(f"{'='*45}")

    print("Training records:", len(train))
    print("Validation records:", len(validation))
    print("Testing records:", len(test))

    print(
        "Training articles:",
        train["article_id"].nunique()
    )

    print(
        "Validation articles:",
        validation["article_id"].nunique()
    )

    print(
        "Testing articles:",
        test["article_id"].nunique()
    )


1-WEEK FORECASTING
Training records: 10758
Validation records: 3467
Testing records: 2051
Training articles: 372
Validation articles: 254
Testing articles: 198

4-WEEK FORECASTING
Training records: 9674
Validation records: 3307
Testing records: 2051
Training articles: 347
Validation articles: 253
Testing articles: 198

8-WEEK FORECASTING
Training records: 8328
Validation records: 3058
Testing records: 2051
Training articles: 320
Validation articles: 248
Testing articles: 198


In [11]:
for horizon, splits in forecast_splits.items():

    print(f"\n{horizon}-Week Forecast")

    for split_name, split_data in splits.items():

        print(f"\n{split_name.upper()}")

        if len(split_data) > 0:

            print(
                "Earliest target:",
                split_data["target_week"].min()
            )

            print(
                "Latest target:",
                split_data["target_week"].max()
            )

        else:

            print("No records available.")


1-Week Forecast

TRAIN
Earliest target: 2018-11-19 00:00:00
Latest target: 2020-02-17 00:00:00

VALIDATION
Earliest target: 2020-02-24 00:00:00
Latest target: 2020-06-08 00:00:00

TEST
Earliest target: 2020-06-15 00:00:00
Latest target: 2020-09-21 00:00:00

4-Week Forecast

TRAIN
Earliest target: 2018-12-10 00:00:00
Latest target: 2020-02-17 00:00:00

VALIDATION
Earliest target: 2020-02-24 00:00:00
Latest target: 2020-06-08 00:00:00

TEST
Earliest target: 2020-06-15 00:00:00
Latest target: 2020-09-21 00:00:00

8-Week Forecast

TRAIN
Earliest target: 2019-01-07 00:00:00
Latest target: 2020-02-17 00:00:00

VALIDATION
Earliest target: 2020-02-24 00:00:00
Latest target: 2020-06-08 00:00:00

TEST
Earliest target: 2020-06-15 00:00:00
Latest target: 2020-09-21 00:00:00


In [12]:
for horizon, df in forecast_datasets.items():

    duplicates = df.duplicated(
        subset=["article_id", "week_start"]
    ).sum()

    print(
        f"{horizon}-week forecast duplicate examples:",
        duplicates
    )

1-week forecast duplicate examples: 0
4-week forecast duplicate examples: 0
8-week forecast duplicate examples: 0


In [13]:
OUTPUT_PATH = Path("../data/processed")

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

for horizon, splits in forecast_splits.items():

    for split_name, split_data in splits.items():

        filename = (
            f"{split_name}_{horizon}week_forecast.csv"
        )

        output_file = OUTPUT_PATH / filename

        split_data.to_csv(
            output_file,
            index=False
        )

        print("Saved:", output_file)

print("\nAll forecasting datasets exported successfully!")

Saved: ..\data\processed\train_1week_forecast.csv
Saved: ..\data\processed\validation_1week_forecast.csv
Saved: ..\data\processed\test_1week_forecast.csv
Saved: ..\data\processed\train_4week_forecast.csv
Saved: ..\data\processed\validation_4week_forecast.csv
Saved: ..\data\processed\test_4week_forecast.csv
Saved: ..\data\processed\train_8week_forecast.csv
Saved: ..\data\processed\validation_8week_forecast.csv
Saved: ..\data\processed\test_8week_forecast.csv

All forecasting datasets exported successfully!


In [14]:
for horizon, df in forecast_datasets.items():

    expected_target = (
        df["week_start"]
        + pd.to_timedelta(7 * horizon, unit="D")
    )

    mismatches = (
        df["target_week"] != expected_target
    ).sum()

    print(f"\n{horizon}-week horizon")
    print("Target date mismatches:", mismatches)

    if mismatches == 0:
        print("Horizon alignment verified!")
    else:
        print("WARNING: Check target date alignment!")


1-week horizon
Target date mismatches: 0
Horizon alignment verified!

4-week horizon
Target date mismatches: 0
Horizon alignment verified!

8-week horizon
Target date mismatches: 0
Horizon alignment verified!
